# LLM Prompt Benchmark Walkthrough

This notebook mirrors the homework workflow from the LLM chapter. Use it to inspect prompts, count tokens, run a prompt-only baseline, run one decoding ablation, and organize the scoring notes.

For repeatable command-line runs, dependency checks, and smoke tests, use `llm_prompt_benchmark.py` in this directory. This notebook calls that script instead of duplicating the benchmark implementation.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Locate the Script and Set Run Defaults

Run this notebook from `chapter_from_transformer_to_llms` or from the repository root. Leave `RUN_GENERATION = False` until your environment can load the selected model.

In [ ]:
from __future__ import annotations

import ast
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


def run_command(args: list[str]) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(args))
    return subprocess.run(args, check=True, text=True)


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def read_jsonl(path: Path) -> list[dict[str, object]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def show_rows(rows: list[dict[str, object]], columns: list[str] | None = None) -> None:
    if not rows:
        print("No rows to display.")
        return
    try:
        import pandas as pd
        from IPython.display import display

        frame = pd.DataFrame(rows)
        if columns is not None:
            frame = frame[[column for column in columns if column in frame.columns]]
        display(frame)
    except Exception:
        print(json.dumps(rows, indent=2)[:4000])


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


NOTEBOOK_DIR = find_chapter_dir("llm_prompt_benchmark.py", "chapter_from_transformer_to_llms")
os.chdir(NOTEBOOK_DIR)
NOTEBOOK_DIR = Path.cwd()
SCRIPT = NOTEBOOK_DIR / "llm_prompt_benchmark.py"
PROMPT_PATH = Path("sample_data/validation_prompts.jsonl")
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RUN_GENERATION = False

print("Working directory:", NOTEBOOK_DIR)
print("Python:", sys.version.split()[0])

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 2. Check the Script and Optional Dependencies

These checks are safe on a CPU-only machine. They parse the script without writing bytecode and report whether the optional runtime packages are installed.

In [ ]:
ast.parse(SCRIPT.read_text(encoding="utf-8"), filename=str(SCRIPT))
run_command([
    sys.executable,
    str(SCRIPT),
    "--check-deps",
    "--allow-missing-deps",
])

## 3. Write Held-Out Validation Prompts

Replace these examples with your own held-out course-assistant prompts before reporting homework results. Keep the same prompt set when comparing decoding settings.

In [ ]:
records = [
    {
        "question": "How do I improve validation accuracy in an image classifier?",
        "reference_answer": (
            "Check the validation split and baseline first. Then change one "
            "factor at a time, such as augmentation, learning rate, model size, "
            "or input resolution. Keep the final test set untouched."
        ),
    },
    {
        "question": "Why should I not tune hyperparameters on the final test set?",
        "reference_answer": (
            "The final test set should estimate performance after model selection. "
            "Repeated tuning on it turns it into another validation set."
        ),
    },
    {
        "question": "What does tokenization change before text enters an LLM?",
        "reference_answer": (
            "Tokenization maps text into token IDs from the model vocabulary. "
            "It changes sequence length, context usage, and what pieces of text "
            "the model can condition on directly."
        ),
    },
    {
        "question": "When should I prefer greedy decoding over top-p sampling?",
        "reference_answer": (
            "Use greedy decoding for a stable baseline and easier comparisons. "
            "Use top-p sampling when controlled variation is useful, and report "
            "the temperature, top-p value, and random seed."
        ),
    },
    {
        "question": "What does the KV cache speed up during autoregressive generation?",
        "reference_answer": (
            "The KV cache stores keys and values for previous tokens, so the "
            "model does not recompute the full prefix at every generation step."
        ),
    },
]

assert len(records) >= 5, "Use at least five held-out prompts."

PROMPT_PATH.parent.mkdir(parents=True, exist_ok=True)
with PROMPT_PATH.open("w", encoding="utf-8") as handle:
    for row in records:
        handle.write(json.dumps(row, sort_keys=True) + "\n")

print(f"Wrote {len(records)} prompts to {PROMPT_PATH}")

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## 4. Preview the Prompt Text

Previewing is dependency-free and catches formatting mistakes before any model or tokenizer is loaded.

In [ ]:
run_command([
    sys.executable,
    str(SCRIPT),
    "--preview",
    "--prompts",
    str(PROMPT_PATH),
])

## 5. Count Tokens with the Selected Tokenizer

This step loads the tokenizer for the selected checkpoint. Token counts are part of the run record because prompt length affects both quality and speed.

In [ ]:
token_counts_path = Path("runs/token_counts.jsonl")

if not package_available("transformers"):
    print("Skipping token counts: install transformers before running this cell.")
else:
    run_command([
        sys.executable,
        str(SCRIPT),
        "--estimate-tokens",
        "--prompts",
        str(PROMPT_PATH),
        "--model-name",
        MODEL_NAME,
        "--use-chat-template",
        "--output",
        str(token_counts_path),
    ])

The next cell reads the token-count artifact back from disk. Reading the file, instead of only trusting in-memory variables, matches the evidence path used by the command-line workflow.

In [ ]:
if token_counts_path.exists():
    show_rows(read_jsonl(token_counts_path), [
        "index",
        "question",
        "prompt_format",
        "prompt_token_count",
    ])
else:
    print(f"No token-count artifact found at {token_counts_path}.")

## 6. Run the Greedy Prompt-Only Baseline

Set `RUN_GENERATION = True` in the first code cell after your runtime can load the selected model. This baseline keeps the model, prompts, prompt format, and decoding settings fixed.

In [ ]:
greedy_outputs = Path("runs/greedy_outputs.jsonl")
greedy_metadata = Path("runs/greedy_metadata.json")

if not RUN_GENERATION:
    print("Skipping generation. Set RUN_GENERATION = True in the first code cell to run this baseline.")
elif not all(package_available(name) for name in ["torch", "transformers", "accelerate"]):
    print("Skipping generation: install torch, transformers, and accelerate first.")
else:
    run_command([
        sys.executable,
        str(SCRIPT),
        "--generate",
        "--prompts",
        str(PROMPT_PATH),
        "--model-name",
        MODEL_NAME,
        "--output",
        str(greedy_outputs),
        "--metadata-output",
        str(greedy_metadata),
        "--max-new-tokens",
        "128",
        "--use-chat-template",
        "--temperature",
        "0",
    ])

## 7. Run One Decoding Ablation

This ablation changes only the decoding rule. The model, tokenizer, prompts, chat-template setting, and maximum-new-token limit stay fixed.

In [ ]:
top_p_outputs = Path("runs/top_p_outputs.jsonl")
top_p_metadata = Path("runs/top_p_metadata.json")

if not RUN_GENERATION:
    print("Skipping generation. Set RUN_GENERATION = True in the first code cell to run this ablation.")
elif not all(package_available(name) for name in ["torch", "transformers", "accelerate"]):
    print("Skipping generation: install torch, transformers, and accelerate first.")
else:
    run_command([
        sys.executable,
        str(SCRIPT),
        "--generate",
        "--prompts",
        str(PROMPT_PATH),
        "--model-name",
        MODEL_NAME,
        "--output",
        str(top_p_outputs),
        "--metadata-output",
        str(top_p_metadata),
        "--max-new-tokens",
        "128",
        "--use-chat-template",
        "--temperature",
        "0.7",
        "--top-p",
        "0.9",
    ])

## 8. Inspect Generated Answers and Cost Fields

Use the generated answers, token counts, first-token latency, throughput, and metadata files as evidence in the report.

In [ ]:
for run_name, output_path in [
    ("greedy", greedy_outputs),
    ("top_p", top_p_outputs),
]:
    print(f"\n{run_name}: {output_path}")
    if output_path.exists():
        show_rows(read_jsonl(output_path), [
            "index",
            "question",
            "prompt_token_count",
            "generated_token_count",
            "first_token_seconds",
            "output_tokens_per_second",
            "generated_text",
        ])
    else:
        print("No output artifact found yet.")

## 9. Prepare the Scoring Table

Fill the rubric fields after reading each answer. The total is out of ten points: technical correctness 4, course terminology 2, applied advice 2, concision 1, and safety 1.

In [ ]:
score_rows = []
for run_name, output_path in [
    ("greedy", greedy_outputs),
    ("top_p", top_p_outputs),
]:
    if not output_path.exists():
        continue
    for row in read_jsonl(output_path):
        score_rows.append({
            "run": run_name,
            "index": row["index"],
            "technical_correctness_0_to_4": None,
            "course_terminology_0_to_2": None,
            "applied_advice_0_to_2": None,
            "concision_0_to_1": None,
            "safety_0_to_1": None,
            "notes": "",
        })

show_rows(score_rows)

## 10. Report Artifacts

These are the files to cite in the homework report after the token-count and generation cells have run.

In [ ]:
artifact_paths = [
    token_counts_path,
    greedy_outputs,
    greedy_metadata,
    top_p_outputs,
    top_p_metadata,
]

for path in artifact_paths:
    status = "exists" if path.exists() else "missing"
    print(f"{path}: {status}")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.